# Gap analysis — is green cover where Bengaluru needs it?

This notebook builds a **500 m grid** over the city and, for every square, calculates:
- `lst` — average land surface temperature (°C), Jan–Apr 2022–2026 (same window as `site_selection`)
- `ndvi` — average greenness
- `green` — green cover: share of the square's 30 m pixels with NDVI > 0.4 (vegetated)
- `water` — share that is water (Dynamic World 2025; water squares are left out later)
- `built` — built-up share (GHSL 2025)
- `pop_density` — people per km² (WorldPop 2020)

It saves one CSV to Google Drive. Download it and put it in `data/`, then run `supporting_green_vs_heat/gap_analysis.py`.

**Run all cells in order.** The export takes about 5–15 minutes after the last cell.

*v2 changes:* population now from WorldPop (GHS-POP 2025 showed near-zero population across the south and east of the city — a data artifact); green cover now from NDVI (Dynamic World labelled most street/garden trees as built-up, giving 0% green in 46% of squares).

# SETUP

In [ ]:
import ee
import geemap.core as geemap

In [ ]:
PROJECT_ID = 'still-algebra-425406-v5'

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project=PROJECT_ID)

In [ ]:
!pip install -q osmnx

## City boundary
Same boundary as the teammates' notebooks (OpenStreetMap, ~717 km²).

In [ ]:
import osmnx as ox
from shapely.geometry import mapping

city_gdf = ox.geocode_to_gdf('Bengaluru, Karnataka, India')
city = ee.Geometry(mapping(city_gdf.geometry.iloc[0].simplify(0.0005)))
print('City area (km2):', round(city.area(1).getInfo() / 1e6))   # should be ~717

## Settings

In [ ]:
YEARS = (2022, 2026)   # dry seasons used
MONTHS = (1, 4)        # Jan-Apr
MAX_CLOUD = 60         # scene-level cloud %
CELL = 500             # grid size in metres

## Heat and greenness from Landsat 8/9

In [ ]:
def prep(img):
    qa = img.select('QA_PIXEL')
    # remove dilated cloud (bit 1), cloud (bit 3), cloud shadow (bit 4)
    clear = (qa.bitwiseAnd(1 << 1).eq(0)
             .And(qa.bitwiseAnd(1 << 3).eq(0))
             .And(qa.bitwiseAnd(1 << 4).eq(0)))
    red = img.select('SR_B4').multiply(0.0000275).add(-0.2)
    nir = img.select('SR_B5').multiply(0.0000275).add(-0.2)
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('ndvi')
    return lst.addBands(ndvi).updateMask(clear).updateMask(img.select('ST_B10').gt(0))

landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(city)
           .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[1] + 1}-01-01')
           .filter(ee.Filter.calendarRange(MONTHS[0], MONTHS[1], 'month'))
           .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD)))
print('Landsat scenes used:', landsat.size().getInfo())

comp = landsat.map(prep).median()

## Green cover, water, built-up and population
Green cover = NDVI above 0.4 (for reference: city median ≈ 0.3, Cubbon Park / Lalbagh ≈ 0.55).

In [ ]:
GREEN_NDVI = 0.4
green = comp.select('ndvi').gt(GREEN_NDVI).rename('green')

dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
      .filterBounds(city).filterDate('2025-01-01', '2026-01-01')
      .select('label').mode())
water = dw.eq(0).rename('water')

blt = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S/2025').select('built_surface')
built = blt.divide(ee.Image.pixelArea().reproject(blt.projection())).rename('built')

# WorldPop: people per 100 m pixel. Summed per square below.
people = (ee.ImageCollection('WorldPop/GP/100m/pop')
          .filter(ee.Filter.eq('country', 'IND')).filter(ee.Filter.eq('year', 2020))
          .first().select('population'))

layers = ee.Image.cat([comp.select('lst'), comp.select('ndvi'), green, water, built]).toFloat()

## 500 m grid
Squares are kept only if their centre lies inside the city.

In [ ]:
grid = city.coveringGrid(ee.Projection('EPSG:32643').atScale(CELL))

def add_centre(cell):
    c = cell.geometry().centroid(1)
    xy = c.coordinates()
    return cell.set({'lon': xy.get(0), 'lat': xy.get(1), 'inside': city.contains(c, 1)})

grid = grid.map(add_centre).filter(ee.Filter.eq('inside', True))
print('Grid squares:', grid.size().getInfo())   # roughly 2,800-2,900

## Quick look (check that the layers look sensible)

In [ ]:
Map = geemap.Map()
Map.centerObject(city, 11)
Map.addLayer(comp.select('lst').clip(city),
             {'min': 28, 'max': 45, 'palette': ['040274', '30c8e2', '86e26e', 'ffff2b', 'ff9c2b', 'ff0000']},
             'LST (°C)')
Map.addLayer(green.clip(city).selfMask(), {'palette': ['1a9850']}, 'Green cover (NDVI > 0.4)', False)
Map.addLayer(people.clip(city), {'min': 0, 'max': 300, 'palette': ['ffffcc', 'fd8d3c', '800026']},
             'Population (WorldPop, per 100 m pixel)', False)
Map.addLayer(grid.style(color='000000', fillColor='00000000', width=0.5), {}, '500 m grid', False)
Map

## Average every layer per square and export to Drive

In [ ]:
stats = layers.reduceRegions(collection=grid, reducer=ee.Reducer.mean(), scale=30, tileScale=4)
stats = people.reduceRegions(collection=stats, reducer=ee.Reducer.sum().setOutputs(['people']),
                             scale=100, tileScale=4)
stats = stats.map(lambda f: f.set('pop_density', ee.Number(f.get('people')).divide(0.25)))   # square = 0.25 km2

task = ee.batch.Export.table.toDrive(
    collection=stats,
    description='bengaluru_grid_500m_v2',
    fileFormat='CSV',
    selectors=['lon', 'lat', 'lst', 'ndvi', 'green', 'water', 'built', 'pop_density'])
task.start()
print('Export started. It will appear in your Google Drive as bengaluru_grid_500m_v2.csv')

In [ ]:
# Re-run this cell to check progress (READY -> RUNNING -> COMPLETED)
print(task.status()['state'], task.status().get('error_message', ''))